In [1]:
import pandas as pd 

In [10]:
df1=pd.read_csv(r"D:\Airbnb Project\listings.csv")

In [7]:
# To remove completely empty columns 
df=df.dropna(how='all',axis=1)

In [4]:
# To grab all the columns that do not belong to the list of columns, we are about to unpivot/melt.
id_vars=[col for col in df.columns if col not in ['availability_30','availability_60','availability_90','availability_365']]

In [15]:
df.to_csv(r"D:\Airbnb Project\listings_cleaned_checkpoint.csv", index=False)

In [15]:
# Before we unpivot the columns, we'd first clean the anchor columns to prevent null values multiplying and filling the data.
fill_values={
    'bathrooms':df['bathrooms'].median(),
    'bedrooms':df['bedrooms'].median(),
    'beds':df['beds'].median()
}
df=df.fillna(value=fill_values)

In [16]:
df['price']=df['price'].str.replace('$','',regex=False).str.replace(',','',regex=False)

In [17]:
df['price']=df['price'].astype(float)

In [18]:
df['price']=df['price'].fillna(df['price'].median())

In [5]:
df_long=pd.melt(
    df,
    id_vars=id_vars,
    value_vars=['availability_30', 'availability_60', 'availability_90', 'availability_365'],
    var_name='availability_window',
    value_name='days_available'
)

In [7]:
df_long['availability_window']=df_long['availability_window'].str.extract(r'(\d+)').astype(int)


In [11]:
df_availability = df_long[['id', 'availability_window', 'days_available']].copy() 

In [41]:
# Saving the lean availability fact table to a CSV
df_availability.to_csv(r"D:\Airbnb Project\fact_availability.csv", index=False)

In [18]:
# Now we'd clear the noise of our base table by removing the availability columns and the over descriptive text columns.
columns_to_drop=['availability_30', 'availability_60', 'availability_90', 'availability_365', # Moved 
                 'listing_url','description','picture_url','host_url','host_thumbnail_url','host_picture_url','host_profile_url' ]
df_base=df.drop(columns=columns_to_drop,errors='ignore')
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16107 entries, 0 to 16106
Data columns (total 68 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            16107 non-null  int64  
 1   scrape_id                                     16107 non-null  int64  
 2   last_scraped                                  16107 non-null  object 
 3   source                                        16107 non-null  object 
 4   name                                          16107 non-null  object 
 5   host_id                                       16107 non-null  int64  
 6   host_profile_id                               15982 non-null  float64
 7   host_name                                     15982 non-null  object 
 8   hosts_time_as_user_years                      15982 non-null  float64
 9   hosts_time_as_user_months                     15982 non-null 

In [50]:
# Building the host dimension 

host_descriptive_columns = [
    'host_id', 'host_profile_id', 'host_name', 
    'hosts_time_as_user_years', 'hosts_time_as_user_months',
    'hosts_time_as_host_years', 'hosts_time_as_host_months',
    'host_location', 'host_is_superhost', 'host_identity_verified'
]

In [51]:
df_host=df_base[host_descriptive_columns].drop_duplicates(subset=['host_id']).copy()

In [19]:
# Just realised putting hosts_time_as column that constantly changes with time is not the best to be put in dimensional table.
host_descriptive_columns = [
    'host_id', 'host_profile_id', 'host_name', 
    'host_location', 'host_is_superhost', 'host_identity_verified'
]

In [54]:
df_host = df_base[host_descriptive_columns].drop_duplicates(subset=['host_id']).copy()

In [57]:
# Removing the unwanted columns
df_host=df_host.drop(columns=['host_profile_id'],errors='ignore')

In [58]:
# Checking for nulls 
print(f"Missing locations: {df_host['host_location'].isnull().sum()}")

Missing locations: 1404


In [5]:
# Assigning appropriate data types to str columns to prevent memory overhead and fast operations.
string_cols=['host_name','host_location','host_is_superhost','host_identity_verified']
df_host[string_cols]=df_host[string_cols].astype('string')

In [11]:
# To check for boolean variations for the columns 'host_is_superhost','host_identity_verified', to bring them to a common value.
print('Superhost different values:',df_host['host_is_superhost'].value_counts(dropna=False))
print('Identity Verified:',df_host['host_identity_verified'].value_counts(dropna=False))

Superhost different values: host_is_superhost
f    3839
t    1399
Name: count, dtype: Int64
Identity Verified: host_identity_verified
t    4493
f     745
Name: count, dtype: Int64


In [62]:
df_host.to_csv(r"D:\Airbnb Project\dim_host_checkpoint.csv", index=False)

In [3]:
df_host=pd.read_csv(r"D:\Airbnb Project\dim_host_checkpoint.csv")

In [10]:
df_host['host_is_superhost']=df_host['host_is_superhost'].fillna('f')
df_host['host_identity_verified']=df_host['host_identity_verified'].fillna('f')

In [14]:
pd.set_option('display.max.rows',100)

In [19]:
# Cleaning the host_name and host_location columns. 
clean_name=df_host['host_name'].str.lower().str.strip() 
# 1. The known pseudo nulls.
known_nulls=r'^(na|unknown|not known|none|-|null|no|not)$'
is_known_null=clean_name.str.match(known_nulls,na=False)
# 2. Only punctuations 
is_pure_punctuation=clean_name.str.match(r'^[^a-z0-9]*$',na=False)
# 3. Too short 
is_too_short=clean_name.str.len()<=1 

all_trash_conditions=is_known_null | is_pure_punctuation | is_too_short 
df_host.loc[all_trash_conditions,'host_name']=pd.NA

In [20]:
df_host['host_name']=df_host['host_name'].fillna('Unknown Host')

In [ ]:
# Host dimension is clean and ready to be saved.
df_host.to_csv(r"D:\Airbnb Project\host_dimension.csv", index=False)

In [23]:
# Cleaning the base table. 
df_host=pd.read_csv("D:\Airbnb Project\listings_dim_host.csv")

<>:2: SyntaxWarning: invalid escape sequence '\A'
<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\rishabh hinduja\AppData\Local\Temp\ipykernel_23680\1340170289.py:2: SyntaxWarning: invalid escape sequence '\A'
  df_host=pd.read_csv("D:\Airbnb Project\listings_dim_host.csv")


In [27]:
columns_to_drop=['host_name','host_location','host_is_superhost','host_identity_verified','host_profile_id']
df_base=df_base.drop(columns=columns_to_drop,errors='ignore')

In [9]:
df_base.to_csv(r"D:\Airbnb Project\base_checkpoint.csv")

In [4]:
df_base=pd.read_csv(r"D:\Airbnb Project\base_checkpoint.csv")

In [7]:
# Making host fact table
host_fact_columns=['host_id', 
                   'last_scraped',
                   'hosts_time_as_user_years',
                   'hosts_time_as_user_months',
                   'hosts_time_as_host_years',
                   'hosts_time_as_host_months',
                   'host_listings_count',
                   'calculated_host_listings_count',
                   'calculated_host_listings_count_entire_homes',
                   'calculated_host_listings_count_private_rooms',
                   'calculated_host_listings_count_shared_rooms']

In [8]:
df_host_fact=df_base[host_fact_columns].copy()

In [12]:
df_host_fact=df_host_fact.drop_duplicates(subset=['host_id','last_scraped'])

In [15]:
# Filling null values with median..why? coz thats safer than mean in case of outliers and 
# makes way more sense than using 0. 
fill_values={
    'hosts_time_as_user_years':df_host_fact['hosts_time_as_user_years'].median(),
    'hosts_time_as_user_months':df_host_fact['hosts_time_as_user_months'].median(),
    'hosts_time_as_host_years':df_host_fact['hosts_time_as_host_years'].median(),
    'hosts_time_as_host_months':df_host_fact['hosts_time_as_host_months'].median(),
    'host_listings_count':df_host_fact['host_listings_count'].median()
}
df_host_fact=df_host_fact.fillna(value=fill_values)

In [17]:
df_host_fact.to_csv(r'D:\Airbnb Project\listings_fact_host.csv')

In [20]:
drop_columns=['hosts_time_as_user_years',
              'hosts_time_as_user_months',
              'hosts_time_as_host_years',
              'hosts_time_as_host_months',
              'host_listings_count',
              'calculated_host_listings_count',
              'calculated_host_listings_count_entire_homes',
              'calculated_host_listings_count_private_rooms',
              'calculated_host_listings_count_shared_rooms']
df_base=df_base.drop(columns=drop_columns,errors='ignore')

In [26]:
# More cleaning 
clutter=['Unnamed: 0','scrape_id','host_about','host_has_profile_pic','has_availability']
df_base=df_base.drop(columns=clutter)

In [30]:
df_neighbourhood=df_base[['neighbourhood_cleansed',
 'neighbourhood_group_cleansed']].drop_duplicates()

In [32]:
df_neighbourhood=df_neighbourhood.reset_index(drop=True)

In [34]:
df_neighbourhood['neighbourhood_id']=df_neighbourhood.index+1

In [39]:
df_base=df_base.merge(df_neighbourhood, 
              on=['neighbourhood_cleansed','neighbourhood_group_cleansed'] )

In [41]:
df_base=df_base.drop(columns=['neighbourhood_cleansed','neighbourhood_group_cleansed'])

In [42]:
df_neighbourhood.to_csv(r"D:\Airbnb Project\listings_dim_neighbourhood.csv")

In [54]:
df_base['bathrooms']=df_base['bathrooms_text'].str.extract(r'(\d+\.?\d*)')[0].astype(float)

In [84]:
# Saving progress before the final imputation
df_base.to_csv(r"D:\Airbnb Project\base_checkpoint_2.csv", index=False)

In [2]:
df_base=pd.read_csv(r"D:\Airbnb Project\base_checkpoint_2.csv")

In [4]:
half_bath_mask=df_base['bathrooms_text'].str.lower().str.contains('half',na=False)
df_base.loc[half_bath_mask,'bathrooms']=0.5

In [12]:
fill_cols=[ 'bathrooms','bedrooms','beds']
for col in fill_cols:
    df_base[col]=df_base[col].fillna(
        df_base.groupby(['room_type','accommodates'])[col].transform('median')
    )

In [13]:
df_base=df_base.drop(columns=['bathrooms_text'])

In [24]:
df_base['price']=df_base['price_quote_price_per_night']
drop_columns=['price_quote_checkin_date', 
              'price_quote_checkout_date', 
              'price_quote_total_price', 
              'price_quote_price_per_night', 
              'price_quote_raw']
df_base=df_base.drop(columns=drop_columns)

In [8]:
df_base=df_base.drop(columns=[
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm'])

In [18]:
df_base=df_base.drop(columns=['calendar_last_scraped'])

In [51]:
df_base=df_base.drop(columns=['number_of_reviews_ltm',
                              'number_of_reviews_l30d',
                              'availability_eoy',
                              'number_of_reviews_ly',
                              'estimated_occupancy_l365d',
                              'estimated_revenue_l365d'])

In [74]:
df_base.columns.tolist()

['id',
 'last_scraped',
 'source',
 'name',
 'host_id',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'number_of_reviews',
 'first_review',
 'last_review',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_value',
 'license',
 'reviews_per_month',
 'neighbourhood_id']

In [69]:
fill_values={
    'bedrooms':df_base['bedrooms'].median(),
    'beds':df_base['beds'].median()
}
df_base=df_base.fillna(value=fill_values)

In [72]:
df_base['price']=df_base['price'].fillna(
    df_base.groupby(['room_type','accommodates'])['price'].transform('median'))

In [77]:
pd.set_option('display.max.columns',31)

In [80]:
pd.set_option('display.max.rows',100) 

In [83]:
df_base[['amenities','license']].head(100)

,amenities,license
0,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Pets allowe...",Spain - National registration number<br />ESFC...
1,"[""Hair dryer"", ""Fire extinguisher"", ""Kitchen"",...",Spain - National registration number<br />ESFC...
2,"[""Hair dryer"", ""Fire extinguisher"", ""Kitchen"",...",Spain - National registration number<br />ESFC...
3,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Elevator"", ...",NaN
4,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Housekeepin...",Barcelona - Regional registration number<br />...
5,"[""Hair dryer"", ""Fire extinguisher"", ""Self chec...",Barcelona - Regional registration number<br />...
6,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Mountain vi...",Barcelona - Regional registration number<br />...
7,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Pets allowe...",Barcelona - Regional registration number<br />...
8,"[""Hair dryer"", ""Babysitter recommendations"", ""...",NaN
9,"[""Hair dryer"", ""Babysitter recommendations"", ""...",Barcelona - Regional registration number<br />...


In [79]:
df_base.head()

,id,last_scraped,source,name,host_id,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,number_of_reviews,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,reviews_per_month,neighbourhood_id
0,18674,2026-03-22,city scrape,Huge flat for 8 people close to Sagrada Familia,71615,41.405560,2.17262,Entire rental unit,Entire home/apt,8,2.0,3.0,6.0,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Pets allowe...",334.00,1.0,999.0,55,2013-05-27,2026-03-02,4.37,4.43,4.57,4.63,4.61,4.81,4.33,Spain - National registration number<br />ESFC...,0.35,1
1,23197,2026-03-21,city scrape,CCIB Forum· Large Balcony · 5 min walk to CCIB,90417,41.412432,2.21975,Entire rental unit,Entire home/apt,5,2.0,3.0,4.0,"[""Hair dryer"", ""Fire extinguisher"", ""Kitchen"",...",452.67,1.0,1125.0,93,2011-03-15,2026-03-06,4.83,4.95,4.90,4.93,4.99,4.67,4.69,Spain - National registration number<br />ESFC...,0.51,2
2,34981,2026-03-21,city scrape,VIDRE HOME PLAZA REAL on LAS RAMBLAS,73163,41.379780,2.17623,Entire rental unit,Entire home/apt,9,3.0,4.0,6.0,"[""Hair dryer"", ""Fire extinguisher"", ""Kitchen"",...",344.64,1.0,1125.0,294,2010-10-03,2026-03-02,4.58,4.63,4.68,4.71,4.74,4.65,4.47,Spain - National registration number<br />ESFC...,1.56,3
3,36763,2026-03-22,previous scrape,In front of the beach,158596,41.380430,2.19094,Private room in rental unit,Private room,1,1.0,1.0,1.0,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Elevator"", ...",42.75,31.0,65.0,108,2011-09-28,2023-12-13,4.79,4.88,4.71,4.93,4.92,4.92,4.82,NaN,0.61,4
4,40983,2026-03-22,city scrape,Soho Colonial Eclectic Apartment,177617,41.396310,2.16832,Entire condo,Entire home/apt,4,1.0,1.0,2.0,"[""Hair dryer"", ""Kitchen"", ""Iron"", ""Housekeepin...",173.46,1.0,364.0,434,2011-06-16,2026-03-13,4.54,4.61,4.58,4.85,4.86,4.90,4.51,Barcelona - Regional registration number<br />...,2.41,5


In [17]:
# Creating a bridge table for exploding amenities there. 
df_amenities_bridge=df_base[['id','amenities']].copy()

In [18]:
df_amenities_bridge['amenities']=df_amenities_bridge['amenities'].str.replace('[','',regex=False)
df_amenities_bridge['amenities']=df_amenities_bridge['amenities'].str.replace(']','',regex=False)
df_amenities_bridge['amenities']=df_amenities_bridge['amenities'].str.replace('"','',regex=False)

In [19]:
df_amenities_bridge['amenities']=df_amenities_bridge['amenities'].str.split(',')

In [20]:
df_amenities_bridge=df_amenities_bridge.explode('amenities')

In [21]:
df_amenities_bridge['amenities'] = df_amenities_bridge['amenities'].str.strip()
df_amenities_bridge = df_amenities_bridge[df_amenities_bridge['amenities'] != ""]

In [22]:
df_amenities_bridge = df_amenities_bridge.dropna(subset=['amenities'])                                  

In [23]:
print(f"Bridge table rows:{len(df_amenities_bridge)}")
display(df_amenities_bridge.head())

Bridge table rows:443860


,id,amenities
0,18674,Hair dryer
0,18674,Kitchen
0,18674,Iron
0,18674,Pets allowed
0,18674,Elevator
